### Data Exploration and Preprocessing

In [1]:
'''
Load the review and metadata files from data/raw for the Electronics Category and comlete EDA on a sample since review size is 18.7M rows.
Used Polars for initial reading in as the file contains 18.3M rows 
Dataset from: https://amazon-reviews-2023.github.io
'''
import polars as pl
import pickle
from pathlib import Path

In [2]:
DATA_DIR = Path("../data/raw")

# use scan_ndjson in polars so we do not load all data immediately into memory -> avoids OOM errors: https://docs.pola.rs/user-guide/io/json/#write
reviews = pl.scan_ndjson(DATA_DIR / "Electronics.jsonl")
product_metadata = pl.scan_ndjson(DATA_DIR / "meta_Electronics.jsonl", schema_overrides = {"price": pl.Utf8}) # Schema parsing happens before transformations in Polars so we Polars tries to read schema and parse float64 right away, so we have to override the schema so Polars treats price as a string before transforming it

# execute the lazy query plan that is currently stored to see example rows for product reviews
print("Example rows from Product Reviews dataframe: ")
reviews.head().collect()

Example rows from Product Reviews dataframe: 


rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
f64,str,str,list[struct[4]],str,str,str,i64,i64,bool
3.0,"""Smells like gasoline! Going ba…","""First & most offensive: they r…","[{""https://m.media-amazon.com/images/I/71YN+Qk3kCL._SL256_.jpg"",""https://m.media-amazon.com/images/I/71YN+Qk3kCL._SL800_.jpg"",""https://m.media-amazon.com/images/I/71YN+Qk3kCL._SL1600_.jpg"",""IMAGE""}]","""B083NRGZMM""","""B083NRGZMM""","""AFKZENTNBQ7A7V7UXW5JJI6UGRYQ""",1658185117948,0,true
1.0,"""Didn’t work at all lenses loos…","""These didn’t work. Idk if they…",[],"""B07N69T6TM""","""B07N69T6TM""","""AFKZENTNBQ7A7V7UXW5JJI6UGRYQ""",1592678549731,0,true
5.0,"""Excellent!""","""I love these. They even come w…",[],"""B01G8JO5F2""","""B01G8JO5F2""","""AFKZENTNBQ7A7V7UXW5JJI6UGRYQ""",1523093017534,0,true
5.0,"""Great laptop backpack!""","""I was searching for a sturdy b…",[],"""B001OC5JKY""","""B001OC5JKY""","""AGGZ357AO26RQZVRLGU4D4N52DZQ""",1290278495000,18,true
5.0,"""Best Headphones in the Fifties…","""I've bought these headphones t…",[],"""B013J7WUGC""","""B07CJYMRWM""","""AG2L7H23R5LLKDKLBEF2Q3L2MVDA""",1676601581238,0,true


In [3]:
# print example rows from product metadata
print("Example rows from Product metadata dataframe: ")
product_metadata.head().collect()

Example rows from Product metadata dataframe: 


main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
str,str,f64,i64,list[str],list[str],str,list[struct[4]],list[struct[3]],str,list[str],struct[165],str,null
"""All Electronics""","""FS-1051 FATSHARK TELEPORTER V3…",3.5,6,[],"[""Teleporter V3 The “Teleporter V3” kit sets a new level of value in the FPV world with Fat Shark renowned performance and quality. The fun of FPV is experienced firsthand through the large screen FPV headset with integrated NexwaveRF receiver technology while simultaneously recording onboard HD footage with the included “PilotHD” camera. The “Teleporter V3” kit comes complete with everything you need to step into the cockpit of your FPV vehicle. We’ve included our powerful 250mW 5.8Ghz transmitter, 25 degree FOV headset (largest QVGA display available), the brand new “PilotHD” camera with live AV out and all the cables, antennas and connectors needed.""]",null,"[{""https://m.media-amazon.com/images/I/41qrX56lsYL._AC_US40_.jpg"",""https://m.media-amazon.com/images/I/41qrX56lsYL._AC_.jpg"",""MAIN"",null}]",[],"""Fat Shark""","[""Electronics"", ""Television & Video"", ""Video Glasses""]","{null,null,null,null,null,null,null,null,null,null,null,null,null,null,""Fatshark"",null,""August 2, 2014"",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null}","""B00MCW7G9M""",null
"""All Electronics""","""Ce-H22B12-S1 4Kx2K Hdmi 4Port""",5.0,1,"[""UPC: 662774021904"", ""Weight: 0.600 lbs""]","[""HDMI In - HDMI Out""]",null,"[{""https://m.media-amazon.com/images/I/31OIMoOW70L._AC_US40_.jpg"",""https://m.media-amazon.com/images/I/31OIMoOW70L._AC_.jpg"",""MAIN"",""https://m.media-amazon.com/images/I/51qxU4Zd5TL._AC_SL1050_.jpg""}, {""https://m.media-amazon.com/images/I/41S98g84d0L._AC_US40_.jpg"",""https://m.media-amazon.com/images/I/41S98g84d0L._AC_.jpg"",""PT01"",""https://m.media-amazon.com/images/I/51HDwazbqNL._AC_SL1050_.jpg""}, … {""https://m.media-amazon.com/images/I/51Ta576VKWL._AC_US40_.jpg"",""https://m.media-amazon.com/images/I/51Ta576VKWL._AC_.jpg"",""PT04"",""https://m.media-amazon.com/images/I/71fWq8+PkfL._AC_SL1050_.jpg""}]",[],"""SIIG""","[""Electronics"", ""Television & Video"", … ""HDMI Cables""]","{null,null,null,null,null,null,null,null,null,null,null,null,""5.3 ounces"",null,""SIIG"",null,""June 3, 2015"",null,""0.83 x 4.17 x 2.05 inches"",""CE-H22B12-S1"",""No"",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null}","""B00YT6XQSE""",null
"""Computers""","""Digi-Tatoo Decal Skin Compatib…",4.5,246,"[""WARNING: Please IDENTIFY MODEL NUMBER on the bottom of your Macbook. Only fits fo

In [4]:
'''
Overview of the dataset: Column names/fields, types, size of each dataset, Example records
'''
print(f"Reviews colums: {reviews.collect_schema().names()}\n")
print(f"Reviews colum types: {reviews.collect_schema().dtypes}\n\n")

print(f"Metadata colums: {product_metadata.collect_schema().names()}")
print(f"Metadata colums: {product_metadata.collect_schema().dtypes}")

# fetch number of rows and columns in the reviews and metadata datasets
num_review_rows = reviews.select(pl.len()).collect().item()
num_review_cols = len(reviews.columns)

num_metadata_rows = product_metadata.select(pl.len()).collect().item()
num_metadata_cols = len(product_metadata.columns)

print(f"Reviews row count: {num_review_rows}, shape: ({num_review_rows, num_review_cols})")
print(f"Metadata row count: {num_metadata_rows}, shape: ({num_metadata_rows, num_metadata_cols})")

print("Example records of Reviews Dataset: ")
print(reviews.fetch(5))

print("Example records of Product Metadata Dataset: ")
product_metadata.fetch(5)

Reviews colums: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']

Reviews colum types: <bound method Schema.dtypes of Schema({'rating': Float64, 'title': String, 'text': String, 'images': List(Struct({'small_image_url': String, 'medium_image_url': String, 'large_image_url': String, 'attachment_type': String})), 'asin': String, 'parent_asin': String, 'user_id': String, 'timestamp': Int64, 'helpful_vote': Int64, 'verified_purchase': Boolean})>


Metadata colums: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together']
Metadata colums: <bound method Schema.dtypes of Schema({'main_category': String, 'title': String, 'average_rating': Float64, 'rating_number': Int64, 'features': List(String), 'description': List(String), 'price': String, 'images': List(Struct({'thumb': String, 'large': String

/var/folders/ks/6zk2yh691g72m6mndxtp0bx40000gn/T/ipykernel_55364/2550683999.py:12: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  num_review_cols = len(reviews.columns)


Reviews row count: 43886944, shape: ((43886944, 10))
Metadata row count: 1610012, shape: ((1610012, 14))
Example records of Reviews Dataset: 
shape: (5, 10)
┌────────┬────────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ rating ┆ title      ┆ text       ┆ images    ┆ … ┆ user_id   ┆ timestamp ┆ helpful_v ┆ verified_ │
│ ---    ┆ ---        ┆ ---        ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ote       ┆ purchase  │
│ f64    ┆ str        ┆ str        ┆ list[stru ┆   ┆ str       ┆ i64       ┆ ---       ┆ ---       │
│        ┆            ┆            ┆ ct[4]]    ┆   ┆           ┆           ┆ i64       ┆ bool      │
╞════════╪════════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 3.0    ┆ Smells     ┆ First &    ┆ [{"https: ┆ … ┆ AFKZENTNB ┆ 165818511 ┆ 0         ┆ true      │
│        ┆ like       ┆ most       ┆ //m.media ┆   ┆ Q7A7V7UXW ┆ 7948      ┆           ┆           │
│        ┆ gasoline!  ┆ offensive: 

/var/folders/ks/6zk2yh691g72m6mndxtp0bx40000gn/T/ipykernel_55364/2550683999.py:15: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  num_metadata_cols = len(product_metadata.columns)
/var/folders/ks/6zk2yh691g72m6mndxtp0bx40000gn/T/ipykernel_55364/2550683999.py:21: DeprecationWarning: `LazyFrame.fetch` is deprecated; use `LazyFrame.collect` instead, in conjunction with a call to `head`.
  print(reviews.fetch(5))
/var/folders/ks/6zk2yh691g72m6mndxtp0bx40000gn/T/ipykernel_55364/2550683999.py:24: DeprecationWarning: `LazyFrame.fetch` is deprecated; use `LazyFrame.collect` instead, in conjunction with a call to `head`.
  product_metadata.fetch(5)


main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
str,str,f64,i64,list[str],list[str],str,list[struct[4]],list[struct[3]],str,list[str],struct[165],str,null
"""All Electronics""","""FS-1051 FATSHARK TELEPORTER V3…",3.5,6,[],"[""Teleporter V3 The “Teleporter V3” kit sets a new level of value in the FPV world with Fat Shark renowned performance and quality. The fun of FPV is experienced firsthand through the large screen FPV headset with integrated NexwaveRF receiver technology while simultaneously recording onboard HD footage with the included “PilotHD” camera. The “Teleporter V3” kit comes complete with everything you need to step into the cockpit of your FPV vehicle. We’ve included our powerful 250mW 5.8Ghz transmitter, 25 degree FOV headset (largest QVGA display available), the brand new “PilotHD” camera with live AV out and all the cables, antennas and connectors needed.""]",null,"[{""https://m.media-amazon.com/images/I/41qrX56lsYL._AC_US40_.jpg"",""https://m.media-amazon.com/images/I/41qrX56lsYL._AC_.jpg"",""MAIN"",null}]",[],"""Fat Shark""","[""Electronics"", ""Television & Video"", ""Video Glasses""]","{null,null,null,null,null,null,null,null,null,null,null,null,null,null,""Fatshark"",null,""August 2, 2014"",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null}","""B00MCW7G9M""",null
"""All Electronics""","""Ce-H22B12-S1 4Kx2K Hdmi 4Port""",5.0,1,"[""UPC: 662774021904"", ""Weight: 0.600 lbs""]","[""HDMI In - HDMI Out""]",null,"[{""https://m.media-amazon.com/images/I/31OIMoOW70L._AC_US40_.jpg"",""https://m.media-amazon.com/images/I/31OIMoOW70L._AC_.jpg"",""MAIN"",""https://m.media-amazon.com/images/I/51qxU4Zd5TL._AC_SL1050_.jpg""}, {""https://m.media-amazon.com/images/I/41S98g84d0L._AC_US40_.jpg"",""https://m.media-amazon.com/images/I/41S98g84d0L._AC_.jpg"",""PT01"",""https://m.media-amazon.com/images/I/51HDwazbqNL._AC_SL1050_.jpg""}, … {""https://m.media-amazon.com/images/I/51Ta576VKWL._AC_US40_.jpg"",""https://m.media-amazon.com/images/I/51Ta576VKWL._AC_.jpg"",""PT04"",""https://m.media-amazon.com/images/I/71fWq8+PkfL._AC_SL1050_.jpg""}]",[],"""SIIG""","[""Electronics"", ""Television & Video"", … ""HDMI Cables""]","{null,null,null,null,null,null,null,null,null,null,null,null,""5.3 ounces"",null,""SIIG"",null,""June 3, 2015"",null,""0.83 x 4.17 x 2.05 inches"",""CE-H22B12-S1"",""No"",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null}","""B00YT6XQSE""",null
"""Computers""","""Digi-Tatoo Decal Skin Compatib…",4.5,246,"[""WARNING: Please IDENTIFY MODEL NUMBER on the bottom of your Macbook. Only fits fo

#### Selection and Justification of Fields for Retrieval

##### Reviews: 

We keep the rating, title, and parent_asin columns in the Reviews dataset. 

The text field was left out since it contains the full user review of the product, which would be very large and expensive to embed for each product. The title column provides a better overview of the review itself without containing the full text so that the review sentiment of products can be retrieved along with the product so users can query for products with a specific critertia such as "great battery life" for example

The title column is more useful since it gives a semantic summary of the full review and can add extra inmportant context that the user can retrieve for. For example, the title column lets users query for short phrases such as "long lasting" and retrieve relevant reviews since the title summarizes the main idea of the product review. 

The rating column is helpful for sentiment analysis and metadata later on. so we can later return results based on the how positive or negative a review is. 

parent_asin is kept since it lets us join the Reviews table with the Metadata table on a common key, so every product review can be linked back to its specific product metadata.  

We dropped columns like text, images, asin, timestamp, and user_id since they do not add value for text-based retrieval systems without adding ambiguity. helpful_vote and verified_purchase were also not included to keep the information stored in embeddings more focused on the semantics of the review itself. 

##### Product Metadata: 

We keep the main_category, title, average_rating, rating_number, features, description, price, store, categories, and parent_asin columns. 

The title, description, features and categories fields are the most important for retrieval since since they contain the main description about the product like what the product is, what it does, and how it is categorized.

main_category and store fields were also kept to provide extra context about the product such as what store holds it and specific details such as the material of the prodict. The average_rating and rating_number are useful as metadata so the user can query products based on popularity or quality. The price field is also included to support queries that contain price range filters.

parent_asin is kept again as a joining key so that we can link the metadata table to each products' review. 

We removed images, videos, bought_together, and details since they are not that useful for direct text based retrieval only and do not provide much extra context unless we want to do something like content recommendation. 


In [5]:
reviews = reviews.select([
    "parent_asin",
    "rating",
    "title"
])

product_metadata = product_metadata.select([
    "parent_asin",
    "main_category",
    "title",
    "average_rating",
    "rating_number",
    "features",
    "description",
    "price",
    "store",
    "categories"
])

#### Text Preprocessing Decisions

We keep preprocessing simple and focus on keeping the original meaning of the product and review text.

For both the reviews and metadata datasets, we lowercase the text, handle null values, and remove extra whitespace. These steps make the text more consistent without removing important meaning. For example, lowercasing ensures that words such as “Battery” and “battery” are treated the same during retrieval, while whitespace normalization helps clean up irregular spacing from raw text fields. We also chose to join the reviews and metadata datasets on the parent_asin key before building the retrieval corpus. This is because every review and product metadata document is stored in different tables so joining them lets every retrieval document contain the product metadata context like the title, description and features, along with the review level context like the review title and review text. This makes the retrieval for the end user more robust as it makes it easier to retrieve both product descriptions and user reviews to user queries. 

For the metadata dataset, some columns such as features, description, categories, and details contain lists of strings. We plan to convert these fields into text strings so they can be included in the retrieval text more cleanly instead of being stored as list data structures.

In [6]:
# preprocess the review field by lowercasing the next, handling null values, and removing extra whitespace 
reviews_clean = reviews.with_columns([
    pl.col("title")
        .fill_null("")
        .str.to_lowercase()
        .str.replace_all(r"\s+", " ") # replace 1+ whitespace chars
        .str.strip_chars()
        .alias("review_title"),
])


# preprocess the metadata fields with the same preprocessing: fill in null values, lowercase them, and replace whitespace chars
product_metadata_clean = product_metadata.with_columns([
    pl.col("main_category")
      .fill_null("")
      .str.to_lowercase()
      .str.replace_all(r"\s+", " ")
      .str.strip_chars(),

    pl.col("title")
      .fill_null("")
      .str.to_lowercase()
      .str.replace_all(r"\s+", " ")
      .str.strip_chars()
      .alias("product_title"),

    pl.col("store")
      .fill_null("")
      .str.to_lowercase()
      .str.replace_all(r"\s+", " ")
      .str.strip_chars(),

    # convert list/string fields to string format
    pl.col("features")
      .list.join(" ")
      .fill_null("")
      .cast(pl.Utf8)
      .str.to_lowercase()
      .str.replace_all(r"\s+", " ")
      .str.strip_chars(),

    pl.col("description")
      .list.join(" ")
      .fill_null("")
      .cast(pl.Utf8)
      .str.to_lowercase()
      .str.replace_all(r"\s+", " ")
      .str.strip_chars(),

    pl.col("categories")
      .list.join(" ")
      .fill_null("")
      .cast(pl.Utf8)
      .str.to_lowercase()
      .str.replace_all(r"\s+", " ")
      .str.strip_chars(), 

    pl.col("price")
      .cast(pl.Utf8, strict = False)
      .fill_null("")
      .str.to_lowercase()
      .str.replace_all(r"\s+", " ")
      .str.strip_chars()
])


In [7]:
print("Example records of Cleaned Review Dataset: ")
print(reviews_clean.head().collect())

Example records of Cleaned Review Dataset: 
shape: (5, 4)
┌─────────────┬────────┬─────────────────────────────────┬─────────────────────────────────┐
│ parent_asin ┆ rating ┆ title                           ┆ review_title                    │
│ ---         ┆ ---    ┆ ---                             ┆ ---                             │
│ str         ┆ f64    ┆ str                             ┆ str                             │
╞═════════════╪════════╪═════════════════════════════════╪═════════════════════════════════╡
│ B083NRGZMM  ┆ 3.0    ┆ Smells like gasoline! Going ba… ┆ smells like gasoline! going ba… │
│ B07N69T6TM  ┆ 1.0    ┆ Didn’t work at all lenses loos… ┆ didn’t work at all lenses loos… │
│ B01G8JO5F2  ┆ 5.0    ┆ Excellent!                      ┆ excellent!                      │
│ B001OC5JKY  ┆ 5.0    ┆ Great laptop backpack!          ┆ great laptop backpack!          │
│ B07CJYMRWM  ┆ 5.0    ┆ Best Headphones in the Fifties… ┆ best headphones in the fifties… │
└───────────

In [8]:
print("Example records of Cleaned Product Metadata Dataset: ")
print(product_metadata_clean.head().collect())

Example records of Cleaned Product Metadata Dataset: 
shape: (5, 11)
┌────────────┬────────────┬────────────┬───────────┬───┬───────┬───────────┬───────────┬───────────┐
│ parent_asi ┆ main_categ ┆ title      ┆ average_r ┆ … ┆ price ┆ store     ┆ categorie ┆ product_t │
│ n          ┆ ory        ┆ ---        ┆ ating     ┆   ┆ ---   ┆ ---       ┆ s         ┆ itle      │
│ ---        ┆ ---        ┆ str        ┆ ---       ┆   ┆ str   ┆ str       ┆ ---       ┆ ---       │
│ str        ┆ str        ┆            ┆ f64       ┆   ┆       ┆           ┆ str       ┆ str       │
╞════════════╪════════════╪════════════╪═══════════╪═══╪═══════╪═══════════╪═══════════╪═══════════╡
│ B00MCW7G9M ┆ all electr ┆ FS-1051    ┆ 3.5       ┆ … ┆       ┆ fat shark ┆ electroni ┆ fs-1051   │
│            ┆ onics      ┆ FATSHARK   ┆           ┆   ┆       ┆           ┆ cs televi ┆ fatshark  │
│            ┆            ┆ TELEPORTER ┆           ┆   ┆       ┆           ┆ sion &    ┆ teleporte │
│            ┆        

In [9]:
# aggregate review titles so we return one grouped value containing joined review titltes to get one review title summary per row 
reviews_grouped =  reviews_clean.group_by("parent_asin").agg([
    pl.col("review_title")
        .drop_nulls()
        .str.join(" ")
        .alias("all_review_titles"),

    # count reviews for each prouduct 
    pl.len().alias("review_count"),

    # compute average rating 
    pl.col("rating").mean().alias("avg_rating")
])

# join the grouped review titles with the product metadata so we get one row per product for embeddingss 
retrieval_df = product_metadata_clean.join(
    reviews_grouped,
    on = "parent_asin",
    how = "left" # if a product has no reviews in the dataset, still include it
)

retrieval_df.head().collect()

parent_asin,main_category,title,average_rating,rating_number,features,description,price,store,categories,product_title,all_review_titles,review_count,avg_rating
str,str,str,f64,i64,str,str,str,str,str,str,str,u32,f64
"""B00MCW7G9M""","""all electronics""","""FS-1051 FATSHARK TELEPORTER V3…",3.5,6,"""""","""teleporter v3 the “teleporter …","""""","""fat shark""","""electronics television & video…","""fs-1051 fatshark teleporter v3…","""recall, recall, recall....... …",5,3.6
"""B00YT6XQSE""","""all electronics""","""Ce-H22B12-S1 4Kx2K Hdmi 4Port""",5.0,1,"""upc: 662774021904 weight: 0.60…","""hdmi in - hdmi out""","""""","""siig""","""electronics television & video…","""ce-h22b12-s1 4kx2k hdmi 4port""","""does what it’s supposed to, sp…",1,5.0
"""B07SM135LS""","""computers""","""Digi-Tatoo Decal Skin Compatib…",4.5,246,"""warning: please identify model…","""""","""19.99""","""digi-tatoo""","""electronics computers & access…","""digi-tatoo decal skin compatib…","""looks nice and provides the pr…",49,4.408163
"""B089CNGZCW""","""amazon fashion""","""NotoCity Compatible with Vivoa…",4.5,233,"""☛notocity 22mm band is designe…","""""","""9.99""","""notocity""","""electronics wearable technolog…","""notocity compatible with vivoa…","""size is good great band for la…",45,4.377778
"""B004E2Z88O""","""cell phones & accessories""","""Motorola Droid X Essentials Co…",3.8,64,"""new droid x essentials combo p…","""all genuine high quality motor…","""14.99""","""verizon""","""electronics computers & access…","""motorola droid x essentials co…","""great product, great purchase,…",40,4.65


In [ ]:
# make one retrieval text document per product with the product title, main category, store, deatures, description, categories, and review titles 
# Docs used for syntax help: https://docs.pola.rs/api/python/dev/reference/expressions/api/polars.lit.html, https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.concat_str.html and https://stackoverflow.com/questions/72400218/how-to-use-polars-concat-str-to-combine-multiple-columns-selected-by-regex
retrieval_df = retrieval_df.with_columns([
    pl.concat_str([ 
        pl.lit("product title: "), pl.col("product_title").fill_null(""),
        pl.lit(". main category: "), pl.col("main_category").fill_null(""),
        pl.lit(". store: "), pl.col("store").fill_null(""),
        pl.lit(". features of product: "), pl.col("features").fill_null(""),
        pl.lit(". description: "), pl.col("description").fill_null(""),
        pl.lit(". categories: "), pl.col("categories").fill_null(""),
        pl.lit(". all review titles: "), pl.col("all_review_titles").fill_null("")
    ], separator = "")
    .str.replace_all(r"\s+", " ") # replace consecutive whitespace chars
    .str.strip_chars() # remove leading and trailing whitespace chars: https://docs.pola.rs/api/python/stable/reference/series/api/polars.Series.str.strip_chars.html
    .alias("retrieval_text")
])

# keep all useful columns that we mentioned above, including retrieval_text (column that will be used for embeddings) and metadata columns such as price
retrieval_corpus = retrieval_df.select([
    "parent_asin",
    "product_title",
    "description",
    "main_category",
    "store",
    "price",
    "average_rating",
    "rating_number",
    "review_count",
    "features",
    "categories",
    "all_review_titles",
    "retrieval_text"
])

retrieval_corpus.select(pl.len()).collect()


# write the lazy frame to a parquet file using sink_parquet(): https://docs.pola.rs/api/python/stable/reference/api/polars.LazyFrame.sink_parquet.html
retrieval_corpus.sink_parquet("../data/processed/retrieval_corpus.parquet")

In [11]:
from langchain_core.documents import Document

# ingest each retrieval text and metadata into a LangChain document. We used metadata to produce better user output and support more diverse user queries such as ones related to price.
# used the docs for help with metadata parsing: https://reference.langchain.com/python/langchain-core/documents/base/Document
documents = []
retrieval_corpus_df = retrieval_corpus.collect()

for row in retrieval_corpus_df.iter_rows(named = True):
    document = Document(
        page_content = row["retrieval_text"],
        metadata = {
            "parent_asin": row["parent_asin"],
            "product_title": row["product_title"],
            "main_category": row["main_category"],
            "description": row["description"],
            "store": row["store"],
            "price": row["price"],
            "average_rating": row["average_rating"],
            "rating_number": row["rating_number"],
            "review_count": row["review_count"],
        }
    )

    documents.append(document)

In [12]:
# confirm a few documents got loaded into the DocumentLoader
print(type(documents[0]))
print(documents[0])

<class 'langchain_core.documents.base.Document'>
page_content='product title: fs-1051 fatshark teleporter v3 headset. main category: all electronics. store: fat shark. features of product: . description: teleporter v3 the “teleporter v3” kit sets a new level of value in the fpv world with fat shark renowned performance and quality. the fun of fpv is experienced firsthand through the large screen fpv headset with integrated nexwaverf receiver technology while simultaneously recording onboard hd footage with the included “pilothd” camera. the “teleporter v3” kit comes complete with everything you need to step into the cockpit of your fpv vehicle. we’ve included our powerful 250mw 5.8ghz transmitter, 25 degree fov headset (largest qvga display available), the brand new “pilothd” camera with live av out and all the cables, antennas and connectors needed.. categories: electronics television & video video glasses. all review titles: recall, recall, recall....... easy to install ... go back a

In [13]:
# print out content vs metadata
print("PAGE CONTENT:\n", documents[0].page_content[:300])
print("\nMETADATA:\n", documents[0].metadata)

PAGE CONTENT:
 product title: fs-1051 fatshark teleporter v3 headset. main category: all electronics. store: fat shark. features of product: . description: teleporter v3 the “teleporter v3” kit sets a new level of value in the fpv world with fat shark renowned performance and quality. the fun of fpv is experienced

METADATA:
 {'parent_asin': 'B00MCW7G9M', 'product_title': 'fs-1051 fatshark teleporter v3 headset', 'main_category': 'all electronics', 'description': 'teleporter v3 the “teleporter v3” kit sets a new level of value in the fpv world with fat shark renowned performance and quality. the fun of fpv is experienced firsthand through the large screen fpv headset with integrated nexwaverf receiver technology while simultaneously recording onboard hd footage with the included “pilothd” camera. the “teleporter v3” kit comes complete with everything you need to step into the cockpit of your fpv vehicle. we’ve included our powerful 250mw 5.8ghz transmitter, 25 degree fov headset (large

In [14]:
# confirm the number of rows from the retrieval_corpus match the length of the documents
print("Number of documents:", len(documents))
print(retrieval_corpus.select(pl.len()).collect())

Number of documents: 1610012
shape: (1, 1)
┌─────────┐
│ len     │
│ ---     │
│ u32     │
╞═════════╡
│ 1610012 │
└─────────┘


In [15]:
# make sure the path we want to save the LangChain docs to exists
Path("../data/processed").mkdir(parents = True, exist_ok = True)

# persist the list of LangChain documents using pickle 
with open("../data/processed/langchain_documents.pkl", "wb") as f:
    pickle.dump(documents, f)